In [2]:
!pip install segmentation-models-pytorch albumentations pyyaml


In [10]:
"""
=============================================================
  TREINO — DETECÇÃO DE PAINÉIS SOLARES
  Arquitetura : UNet + ResNet34 (segmentation_models_pytorch)
  Ambiente    : Kaggle (GPU P100 / T4)
=============================================================

ESTRUTURA DO data.yaml (em /kaggle/working/data.yaml):
    path: /kaggle/input/datasets/patrinnyrocha/datayaml/data.yaml
    train: train/images
    val:   valid/images
    masks: masks          # subpasta de máscaras dentro de train/ e valid/
    nc: 1
    names: ['solar_panel']

SAÍDAS (todas em /kaggle/working/):
    checkpoints/best_iou.pth   ← melhor modelo
    checkpoints/last.pth       ← último epoch
    checkpoints/history.csv    ← métricas por epoch
    checkpoints/config.yaml    ← cópia do CFG usado

DEPENDÊNCIAS:
    pip install segmentation-models-pytorch albumentations pyyaml
"""

import os, time, csv, yaml
import numpy as np
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
from pathlib import Path


# ==============================================================
# SEÇÃO 1 — HIPERPARÂMETROS GLOBAIS
# ✏️  Este é o único bloco que você precisa editar normalmente.
# ==============================================================

CFG = {
    # --- Caminhos Kaggle ---
    # data.yaml fica em /kaggle/working/ (você faz upload junto com o notebook)
    # Todos os resultados são salvos em /kaggle/working/ (persistidos no Kaggle)
    "yaml_path" : "/kaggle/input/datasets/patrinnyrocha/datayaml/data.yaml",
    "save_dir"  : "/kaggle/working",

    # --- Modelo ---
    "encoder"     : "resnet34",   # resnet34 | resnet50 | efficientnet-b3
    "pretrained"  : "imagenet",   # "imagenet" ou None
    "dropout_dec" : 0.2,          # ✏️ suba para 0.3-0.4 se overfitting

    # --- Imagem ---
    # ✏️  ÚNICO valor usado em treino, val e inferência.
    #     Use o mesmo em infer_solar.py → PREPROCESSING_CONFIG["input_size"]
    "img_size" : 512,             # 512 = bom equilíbrio | 640 = mais detalhe (mais lento)

    # --- Treino ---
    "epochs"      : 60,
    "batch_size"  : 8,            # ✏️  reduza para 4 se der OOM (out of memory)
    "num_workers" : 2,

    # --- Otimizador (AdamW com LR diferenciado) ---
    "lr_encoder"  : 1e-4,   # encoder ResNet: LR menor (já pré-treinado)
    "lr_decoder"  : 5e-4,   # decoder UNet: LR maior (treinado do zero)
    "weight_decay": 1e-4,

    # --- Scheduler (warmup linear → cosine annealing) ---
    "warmup_epochs" : 5,      # LR sobe linearmente nos primeiros N epochs
    "cosine_min_lr" : 1e-6,   # LR mínimo ao final do treino

    # --- Loss (FocalDice) ---
    # ✏️  Se muitos falsos positivos em telhados → aumente focal_alpha para 0.85
    # ✏️  Se painéis pequenos não detectados    → aumente focal_gamma para 2.5
    "focal_alpha"  : 0.75,
    "focal_gamma"  : 2.0,
    "dice_weight"  : 0.5,
    "focal_weight" : 0.5,

    # --- WeightedRandomSampler ---
    # ✏️  Aumente se o dataset tiver muito mais imagens sem painel do que com painel
    "sampler_pos_weight": 3.0,

    # --- Early stopping ---
    "patience" : 12,   # para se val IoU não melhorar por N epochs seguidos
}


# ==============================================================
# SEÇÃO 2 — LEITURA DO data.yaml
# ==============================================================

def load_yaml_paths(yaml_path: str) -> dict:
    """
    Lê data.yaml e devolve os caminhos absolutos de treino/val/máscaras.

    Exemplo de data.yaml:
        path: /kaggle/input/datasets/patrinnyrocha/datayaml/data.yaml
        train: train/images
        val:   valid/images
        masks: masks
        nc: 1
        names: ['solar_panel']
    """
    yaml_path = Path(yaml_path)
    if not yaml_path.exists():
        raise FileNotFoundError(
            f"\n❌ data.yaml não encontrado em: {yaml_path}\n"
            f"   Faça upload do arquivo para /kaggle/working/ antes de rodar.\n"
            f"   Conteúdo mínimo esperado:\n"
            f"     path: /kaggle/input/.../dataset_unet\n"
            f"     train: train/images\n"
            f"     val:   valid/images\n"
            f"     masks: masks\n"
        )

    with open(yaml_path) as f:
        data = yaml.safe_load(f)

    base      = Path(data["path"])
    train_img = base / data["train"]                      # ex: .../train/images
    val_img   = base / data["val"]                        # ex: .../valid/images
    masks_sub = data.get("masks", "masks")                # ex: "masks"
    train_msk = train_img.parent / masks_sub              # ex: .../train/masks
    val_msk   = val_img.parent   / masks_sub              # ex: .../valid/masks

    print(f"📂 Dataset raiz : {base}")
    print(f"   Treino imgs  : {train_img}")
    print(f"   Treino masks : {train_msk}")
    print(f"   Val imgs     : {val_img}")
    print(f"   Val masks    : {val_msk}")

    for p in [train_img, train_msk, val_img, val_msk]:
        if not p.exists():
            raise FileNotFoundError(f"❌ Pasta não encontrada: {p}")

    return {
        "train_img": str(train_img),
        "train_msk": str(train_msk),
        "val_img"  : str(val_img),
        "val_msk"  : str(val_msk),
    }


# ==============================================================
# SEÇÃO 3 — DATASET
# ==============================================================

class SolarPanelDataset(Dataset):
    def __init__(self, img_dir: str, mask_dir: str, transform=None):
        self.img_dir   = Path(img_dir)
        self.mask_dir  = Path(mask_dir)
        self.transform = transform

        self.images = sorted([
            p for p in self.img_dir.iterdir()
            if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
        ])
        if not self.images:
            raise FileNotFoundError(f"Nenhuma imagem em: {img_dir}")

    def __len__(self):
        return len(self.images)

    def _find_mask(self, img_path: Path) -> Path:
        for ext in [".png", ".jpg", ".jpeg"]:
            p = self.mask_dir / (img_path.stem + ext)
            if p.exists():
                return p
        raise FileNotFoundError(
            f"Máscara não encontrada para: {img_path.name}\n"
            f"  Procurado em: {self.mask_dir}"
        )

    def __getitem__(self, idx):
        img_path  = self.images[idx]
        mask_path = self._find_mask(img_path)

        image = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        mask  = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        mask  = (mask > 127).astype(np.float32)   # binária [0, 1]

        if self.transform:
            aug   = self.transform(image=image, mask=mask)
            image = aug["image"]
            mask  = aug["mask"].unsqueeze(0)       # (H,W) → (1,H,W)

        return image, mask

    def pos_fraction(self, idx: int) -> float:
        """Fração de pixels positivos — usada para o Sampler."""
        mask_path = self._find_mask(self.images[idx])
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        return float((mask > 127).mean())


# ==============================================================
# SEÇÃO 4 — TRANSFORMS
# ✏️  Treino e val usam o MESMO img_size.
#     Augmentações APENAS no treino.
# ==============================================================

def build_transforms(img_size: int):
    mean = (0.485, 0.456, 0.406)
    std  = (0.229, 0.224, 0.225)

    train_tf = A.Compose([
        A.Resize(img_size, img_size),

        # Geométricas
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Affine(translate_percent=0.03, scale=(0.90, 1.10), rotate=(-20, 20), p=0.5),

        # Cor / iluminação
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=35, val_shift_limit=25, p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.05, p=0.4),

        # ✏️ CLAHE: melhora contraste — ajuda separar painéis de telhados metálicos
        A.CLAHE(clip_limit=2.5, tile_grid_size=(8, 8), p=0.35),

        # ✏️ RandomShadow: simula sombra de árvore/prédio sobre painéis
        A.RandomShadow(shadow_roi=(0, 0, 1, 1), num_shadows_limit=(1, 2), p=0.25),

        # Ruído / blur
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(std_range=(0.01, 0.04), p=0.2),

        # ✏️ CoarseDropout: simula oclusões (antenas, caixas d'água, folhas)
        A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(8, 32),
            hole_width_range=(8, 32),
            fill=0, p=0.2,
        ),

        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

    val_tf = A.Compose([
        A.Resize(img_size, img_size),   # ← mesmo tamanho do treino
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])

    return train_tf, val_tf


# ==============================================================
# SEÇÃO 5 — MODELO
# ==============================================================

def build_model(cfg: dict) -> nn.Module:
    return smp.Unet(
        encoder_name    = cfg["encoder"],
        encoder_weights = cfg["pretrained"],
        in_channels     = 3,
        classes         = 1,
        activation      = None,              # logits brutos; loss aplica sigmoid
        decoder_dropout = cfg["dropout_dec"],
    )


# ==============================================================
# SEÇÃO 6 — LOSS
# ==============================================================

class FocalDiceLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0, dice_w=0.5, focal_w=0.5):
        super().__init__()
        self.focal  = smp.losses.FocalLoss(mode="binary", alpha=alpha, gamma=gamma)
        self.dice   = smp.losses.DiceLoss(mode="binary", smooth=1.0)
        self.dice_w = dice_w
        self.focal_w = focal_w

    def forward(self, logits, targets):
        return self.focal_w * self.focal(logits, targets) \
             + self.dice_w  * self.dice(logits, targets)


# ==============================================================
# SEÇÃO 7 — MÉTRICAS
# ==============================================================

@torch.no_grad()
def compute_metrics(logits, masks, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()
    tp = (preds * masks).sum()
    fp = (preds * (1 - masks)).sum()
    fn = ((1 - preds) * masks).sum()
    iou       = (tp / (tp + fp + fn + 1e-6)).item()
    precision = (tp / (tp + fp + 1e-6)).item()
    recall    = (tp / (tp + fn + 1e-6)).item()
    f1        = 2 * precision * recall / (precision + recall + 1e-6)
    return {"iou": iou, "precision": precision, "recall": recall, "f1": f1}


# ==============================================================
# SEÇÃO 8 — SAMPLER
# ==============================================================

def build_sampler(dataset: SolarPanelDataset, pos_weight: float) -> WeightedRandomSampler:
    print("⚙️  Calculando pesos do sampler (pode demorar alguns segundos)...")
    weights = [
        pos_weight if dataset.pos_fraction(i) > 0.01 else 1.0
        for i in range(len(dataset))
    ]
    n_pos = sum(1 for w in weights if w > 1.0)
    print(f"   Imagens com painel: {n_pos}/{len(dataset)} "
          f"({100*n_pos/len(dataset):.1f}%)")
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


# ==============================================================
# SEÇÃO 9 — SCHEDULER (warmup + cosine)
# ==============================================================

def build_scheduler(optimizer, cfg: dict, steps_per_epoch: int):
    warmup = cfg["warmup_epochs"] * steps_per_epoch
    total  = cfg["epochs"]        * steps_per_epoch

    def lr_lambda(step):
        if step < warmup:
            return step / max(warmup, 1)
        progress = (step - warmup) / max(total - warmup, 1)
        cosine   = 0.5 * (1 + np.cos(np.pi * progress))
        min_frac = cfg["cosine_min_lr"] / cfg["lr_decoder"]
        return max(min_frac, cosine)

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


# ==============================================================
# SEÇÃO 10 — LOOPS DE TREINO E VALIDAÇÃO
# ==============================================================

def train_one_epoch(model, loader, criterion, optimizer, scheduler, device, scaler):
    model.train()
    total_loss, all_iou = 0.0, []
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, masks)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        all_iou.append(compute_metrics(logits.detach(), masks)["iou"])
    return total_loss / len(loader), float(np.mean(all_iou))


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_m = {"iou": [], "precision": [], "recall": [], "f1": []}
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            logits = model(imgs)
            loss   = criterion(logits, masks)
        total_loss += loss.item()
        m = compute_metrics(logits, masks)
        for k in all_m:
            all_m[k].append(m[k])
    return total_loss / len(loader), {k: float(np.mean(v)) for k, v in all_m.items()}


# ==============================================================
# SEÇÃO 11 — SALVAMENTO
# ==============================================================

def save_checkpoint(path: str, model, optimizer, epoch: int, best_iou: float, cfg: dict):
    torch.save({
        "epoch"            : epoch,
        "model_state_dict" : model.state_dict(),
        "optimizer_state"  : optimizer.state_dict(),
        "best_iou"         : best_iou,
        "cfg"              : cfg,
    }, path)


def save_config(save_dir: str, cfg: dict):
    """Salva o CFG usado neste treino para reproduzir depois."""
    out = Path(save_dir) / "config.yaml"
    with open(out, "w") as f:
        yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)
    print(f"   Config salvo em: {out}")


def init_csv(save_dir: str) -> str:
    path = str(Path(save_dir) / "history.csv")
    with open(path, "w", newline="") as f:
        csv.writer(f).writerow([
            "epoch", "tr_loss", "tr_iou",
            "va_loss", "va_iou", "va_precision", "va_recall", "va_f1",
            "lr_decoder",
        ])
    return path


def append_csv(path: str, row: dict):
    with open(path, "a", newline="") as f:
        csv.DictWriter(f, fieldnames=row.keys()).writerow(row)


# ==============================================================
# SEÇÃO 12 — MAIN
# ==============================================================

def main():
    cfg    = CFG
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🖥️  Device : {device}")
    print(f"📄 YAML   : {cfg['yaml_path']}")
    print(f"💾 Output : {cfg['save_dir']}\n")

    os.makedirs(cfg["save_dir"], exist_ok=True)

    # 1. Caminhos via data.yaml
    paths = load_yaml_paths(cfg["yaml_path"])

    # 2. Transforms
    train_tf, val_tf = build_transforms(cfg["img_size"])

    # 3. Datasets
    ds_train = SolarPanelDataset(paths["train_img"], paths["train_msk"], train_tf)
    ds_val   = SolarPanelDataset(paths["val_img"],   paths["val_msk"],   val_tf)
    print(f"\n📦 Treino: {len(ds_train)} | Validação: {len(ds_val)} imagens")

    # 4. Sampler + DataLoaders
    sampler = build_sampler(ds_train, cfg["sampler_pos_weight"])
    loader_train = DataLoader(
        ds_train, batch_size=cfg["batch_size"], sampler=sampler,
        num_workers=cfg["num_workers"], pin_memory=True, drop_last=True,
    )
    loader_val = DataLoader(
        ds_val, batch_size=cfg["batch_size"], shuffle=False,
        num_workers=cfg["num_workers"], pin_memory=True,
    )

    # 5. Modelo
    model = build_model(cfg).to(device)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"🧠 Parâmetros: {n_params:.1f}M  |  img_size: {cfg['img_size']}")

    # 6. Loss, otimizador, scheduler, scaler
    criterion = FocalDiceLoss(
        alpha=cfg["focal_alpha"], gamma=cfg["focal_gamma"],
        dice_w=cfg["dice_weight"], focal_w=cfg["focal_weight"],
    )
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": cfg["lr_encoder"]},
        {"params": [p for n, p in model.named_parameters()
                    if not n.startswith("encoder")], "lr": cfg["lr_decoder"]},
    ], weight_decay=cfg["weight_decay"])
    scheduler = build_scheduler(optimizer, cfg, len(loader_train))
    scaler    = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    # 7. Arquivos de saída
    save_config(cfg["save_dir"], cfg)
    csv_path  = init_csv(cfg["save_dir"])
    best_path = str(Path(cfg["save_dir"]) / "best_iou.pth")
    last_path = str(Path(cfg["save_dir"]) / "last.pth")

    # 8. Loop
    best_iou, patience_cnt = 0.0, 0

    print("\n" + "="*72)
    print(f"{'Ep':>4} {'TrLoss':>8} {'TrIoU':>7} {'VaLoss':>8} "
          f"{'VaIoU':>7} {'Prec':>6} {'Rec':>6} {'F1':>6} {'LR':>9}")
    print("="*72)

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        tr_loss, tr_iou = train_one_epoch(
            model, loader_train, criterion, optimizer, scheduler, device, scaler
        )
        va_loss, va_m = validate(model, loader_val, criterion, device)

        lr_now  = optimizer.param_groups[1]["lr"]
        elapsed = time.time() - t0

        tag = ""
        if va_m["iou"] > best_iou:
            best_iou     = va_m["iou"]
            patience_cnt = 0
            save_checkpoint(best_path, model, optimizer, epoch, best_iou, cfg)
            tag = "  ✅ best"

        save_checkpoint(last_path, model, optimizer, epoch, best_iou, cfg)

        print(
            f"{epoch:>4} {tr_loss:>8.4f} {tr_iou:>7.4f} {va_loss:>8.4f} "
            f"{va_m['iou']:>7.4f} {va_m['precision']:>6.3f} "
            f"{va_m['recall']:>6.3f} {va_m['f1']:>6.3f} "
            f"{lr_now:>9.2e}  [{elapsed:.0f}s]{tag}"
        )

        append_csv(csv_path, {
            "epoch": epoch, "tr_loss": tr_loss, "tr_iou": tr_iou,
            "va_loss": va_loss, "va_iou": va_m["iou"],
            "va_precision": va_m["precision"], "va_recall": va_m["recall"],
            "va_f1": va_m["f1"], "lr_decoder": lr_now,
        })

        if va_m["iou"] <= best_iou and tag == "":
            patience_cnt += 1
            if patience_cnt >= cfg["patience"]:
                print(f"\n⛔ Early stopping — sem melhora por {cfg['patience']} epochs")
                break

    print(f"\n🏁 Concluído. Melhor val IoU : {best_iou:.4f}")
    print(f"   best_iou.pth → {best_path}")
    print(f"   last.pth     → {last_path}")
    print(f"   history.csv  → {csv_path}")


if __name__ == "__main__":
    main()


🖥️  Device : cuda
📄 YAML   : /kaggle/input/datasets/patrinnyrocha/datayaml/data.yaml
💾 Output : /kaggle/working

📂 Dataset raiz : /kaggle/input/datasets/patrinnyrocha/unetdata-equilabrado/unetsolar
   Treino imgs  : /kaggle/input/datasets/patrinnyrocha/unetdata-equilabrado/unetsolar/train/images
   Treino masks : /kaggle/input/datasets/patrinnyrocha/unetdata-equilabrado/unetsolar/train/masks
   Val imgs     : /kaggle/input/datasets/patrinnyrocha/unetdata-equilabrado/unetsolar/valid/images
   Val masks    : /kaggle/input/datasets/patrinnyrocha/unetdata-equilabrado/unetsolar/valid/masks

📦 Treino: 1377 | Validação: 287 imagens
⚙️  Calculando pesos do sampler (pode demorar alguns segundos)...
   Imagens com painel: 836/1377 (60.7%)
🧠 Parâmetros: 24.4M  |  img_size: 512
   Config salvo em: /kaggle/working/config.yaml

  Ep   TrLoss   TrIoU   VaLoss   VaIoU   Prec    Rec     F1        LR
   1   0.4122  0.2549   0.4220  0.2268  0.231  0.857  0.352  1.00e-04  [49s]  ✅ best
   2   0.2691  0.5